# Build a Logistic Regression model

- You've already built a Decision Tree model using the `flights` data. Now you're going to create a Logistic Regression model on the same data.

- The objective is to predict whether a flight is likely to be delayed by at least 15 minutes (label 1) or not (label 0).

- Although you have a variety of predictors at your disposal, you'll only use the `mon`, `depart` and `duration` columns for the moment. These are numerical features which can immediately be used for a Logistic Regression model. You'll need to do a little more work before you can include categorical features. Stay tuned!

- The data have been split into training and testing sets and are available as `flights_train` and `flights_test`.

## Instructions

- Import the class for creating a Logistic Regression classifier.
- Create a classifier object and train it on the training data.
- Make predictions for the testing data and create a confusion matrix.

In [3]:
# # Import the SparkSession class
# import pyspark
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.appName('flight_manipulate_columns').getOrCreate()


In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [2]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [3]:
# Read data from CSV file
flights = spark.read.csv('file:///home/talentum/test-jupyter/c5-MLWithPySpark/M2-Classification/3_LogisticRegression/dataset/flights.csv',
                         sep=',',
                         header=True,
                         inferSchema=True,
                         nullValue='NA')

In [4]:
flights = flights.drop('flight')
flights = flights.dropna()

from pyspark.sql.functions import round

flights = flights.withColumn('km', round(flights.mile * 1.60934, 0))\
.drop('mile')\
.withColumn('label', (flights.delay > 15).cast('integer'))

from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(inputCols=[
'mon', 'depart', 'duration'
], outputCol='features')
flights = assembler.transform(flights)

flights = flights.select('mon', 'depart', 'duration', 'features', 'label')

flights_train, flights_test = flights.randomSplit([0.8, 0.2], seed=43)

print("First few rows from the flights DataFrame:")

flights.show(5, truncate=False)

# Code added by Amit
print(flights_train.printSchema())

First few rows from the flights DataFrame:
+---+------+--------+-----------------+-----+
|mon|depart|duration|features         |label|
+---+------+--------+-----------------+-----+
|0  |16.33 |82      |[0.0,16.33,82.0] |1    |
|2  |6.17  |82      |[2.0,6.17,82.0]  |0    |
|9  |10.33 |195     |[9.0,10.33,195.0]|0    |
|5  |7.98  |102     |[5.0,7.98,102.0] |0    |
|7  |10.83 |135     |[7.0,10.83,135.0]|1    |
+---+------+--------+-----------------+-----+
only showing top 5 rows

root
 |-- mon: integer (nullable = true)
 |-- depart: double (nullable = true)
 |-- duration: integer (nullable = true)
 |-- features: vector (nullable = true)
 |-- label: integer (nullable = true)

None


In [ ]:
# Import the logistic regression class
from pyspark.ml.____ import ____

# Create a classifier object and train on training data
logistic = ____().____(____)

# Create predictions for the testing data and show confusion matrix
prediction = ____.____(____)
prediction.groupBy(____, ____).____().show()

In [5]:
# Import the logistic regression class
from pyspark.ml.classification import LogisticRegression

# Create a classifier object and train on training data
logistic = LogisticRegression().fit(flights_train)
print(type(logistic))

# Create predictions for the testing data and show confusion matrix
prediction = logistic.transform(flights_test)
prediction.groupBy('label', 'prediction').count().show()

# Code added by Amit
print(prediction.printSchema())

<class 'pyspark.ml.classification.LogisticRegressionModel'>
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       0.0| 1856|
|    0|       0.0| 2635|
|    1|       1.0| 2872|
|    0|       1.0| 1981|
+-----+----------+-----+

root
 |-- mon: integer (nullable = true)
 |-- depart: double (nullable = true)
 |-- duration: integer (nullable = true)
 |-- features: vector (nullable = true)
 |-- label: integer (nullable = true)
 |-- rawPrediction: vector (nullable = true)
 |-- probability: vector (nullable = true)
 |-- prediction: double (nullable = false)

None


Now let's unpack that confusion matrix.